# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Resume attempt 2026-10-23T10:00Z: Colab allocation hangs account-wide. Two bounded `colab new -s blb-rice-9f718eb7` attempts (120 s, 420 s) were killed mid-allocation with no session record; `colab sessions` and `colab status -s blb-rice-probe-07c8136d` also hang (exit 124). CLI debug log shows GET/POST /tun/m/assign with no response (one 412) and a GET /tun/m/unassign of stale endpoint m-s-kkb-usw3b1-2rwfg5pfj3d0v hanging since 18:13:18 JST; kernel-ws path still broken. Sibling worker p-8332fc95 was simultaneously waiting on the same hung allocation endpoint; backend demonstrably serves other workers today. Not a notebook defect. No code executed; UNVERIFIED reference notebook preserved at deliverables/p-c02f23ea0e7fc12935556f564c977832.ipynb (outputs empty). Intake re-verified: investigation run 721142 still latest, not superseded. Re-queue with backoff; resume_run_dir once the stale assignment is freed server-side or the CLI regression is fixed.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: A practical framework for rice bacterial blight severity classification using 3D multispectral sensing

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-c02f23ea0e7fc12935556f564c977832)


# Rice bacterial blight severity classification — Table 2 model pipeline (author R code)

**Paper:** S. Baruah et al., *A practical framework for rice bacterial blight severity classification using 3D multispectral sensing*, **Smart Agricultural Technology 15 (2026) 102586**, DOI [10.1016/j.atech.2026.102586](https://doi.org/10.1016/j.atech.2026.102586).

**What this notebook does:** it runs the **authors' own R script** `04_models/run_revised_table2_rf_svm_cv_holdout_models.R` (pinned commit) **unmodified** on the authors' published de-identified input tables. The script performs grouped 5-fold cross-validation on the 90 % training partition (no microplot/unit overlap, SMOTE applied to training data only, parameter tuning inside each outer fold) and a single evaluation on the untouched 10 % holdout test set — the protocol behind the paper's Table 2. Python cells here are only a **launcher** for `Rscript`; all data handling and modelling happen inside R.

**Scope (partial demonstration):** a subset of the Table 2 model suite — the paper's headline configuration, the **4-severity-category Random Forest on binned vegetation indices** — selected via the script's own `MODEL_IDS` mechanism. The full suite (all feature sets × RF/SVM/XGBoost × 4/6 classes) is out of scope for a hands-on run; add more `model_id` values in the selection cell to extend it.

**Runtime:** standard **CPU** Colab runtime is sufficient (no CUDA code is used by the author script; `ranger`/`xgboost`/`e1071` run on CPU). A GPU adds nothing for this method.

**Execution status:** see the *Validation record* section at the bottom. This notebook must be run top-to-bottom with *Runtime → Run all* on a fresh runtime; it writes nothing outside `/content`.

**参考 (Japanese):** このノートブックは論文 Table 2 の作成に用いられた著者自身の R スクリプトを無変更で実行します。Python のセルは `Rscript` の起動のみを行い、データ取得・前処理・学習・評価はすべて R 側で行われます。必要ランタイムは標準 CPU のみです（GPU 不要）。

## How to run / 実行方法

1. Menu **Runtime → Change runtime type → CPU (standard)**, then **Runtime → Run all**. / メニューからランタイムを CPU（標準）に設定し、**ランタイム → すべてのセルを実行** を実行します。
2. Expected total wall time (estimate, not a measured value): ~10–35 min — dominated by nested grouped-CV parameter tuning of the Random Forest (5 outer folds × 9 grid points × 3 inner folds of 1000-tree fits, plus the final model). Package install ≈ 1–3 min; downloads ≈ 7 MB.
3. No GPU, no Drive mount, no credentials. All data is downloaded from the pinned public GitHub commit inside this notebook.
4. Limitation: a single executed configuration is **not** a reproduction of the paper's full Table 2 or of dataset-level benchmark claims; it verifies that the published pipeline runs and yields metrics of the expected magnitude.

**参考:** 所要時間の目安は推定値です。パッケージ導入 1〜3 分、ダウンロード約 7 MB、本処理はネストしたグループ化 CV のため 10〜35 分程度を見込んでください。GPU は不要です。

## Environment and dependency setup / 環境と依存関係

The author script declares its dependencies at the top of the file (commit `63f73a3`, lines 3–9): `data.table`, `e1071`, `FNN`, `ranger`, `xgboost`. We first verify that R is present in the Colab runtime and detect the Ubuntu codename, then install exactly those five packages from the Posit Package Manager **binary** repository matching this VM (fast, no compilation). The printed package versions should be recorded when comparing runs.

**参考:** 著者スクリプトが宣言する 5 つの R パッケージを、この VM の Ubuntu 版コード名に対応した Posit バイナリリポジトリから導入します。まず R の存在とバージョンを確認します。

In [ ]:
import subprocess, pathlib
osr = dict(l.split("=", 1) for l in pathlib.Path("/etc/os-release").read_text().splitlines() if "=" in l)
print("VERSION_CODENAME:", osr.get("VERSION_CODENAME"))
print("Rscript:", subprocess.run(["which", "Rscript"], capture_output=True, text=True).stdout.strip() or "NOT FOUND")
print(subprocess.run(["Rscript", "-e", "cat(R.version.string)"], capture_output=True, text=True).stdout)

Install the five pinned R packages. If a package is already installed, it is skipped; new ones come from the codename-matched P3M binary repo (falls back to CRAN source if the binary repo is unreachable). Versions are printed for the validation record.

**参考:** 依存 5 パッケージを導入し、バージョンを出力します（バイナリ取得で高速化）。

In [ ]:
import subprocess, time, pathlib
codename = ""
for line in pathlib.Path("/etc/os-release").read_text().splitlines():
    if line.startswith("VERSION_CODENAME="):
        codename = line.split("=", 1)[1].strip().strip('"')
repo = f"https://packagemanager.posit.co/cran/__linux__/{codename}/latest"
r_code = f"""
options(repos = c(P3M = "{repo}"), timeout = 600)
pkgs <- c("data.table", "e1071", "FNN", "ranger", "xgboost")
inst <- rownames(installed.packages())
need <- setdiff(pkgs, inst)
if (length(need)) install.packages(need, Ncpus = 2)
for (p in pkgs) cat(p, as.character(packageVersion(p)), "\\n")
"""
t0 = time.time()
r = subprocess.run(["Rscript", "-e", r_code], capture_output=True, text=True, timeout=1500)
print(r.stdout)
if r.returncode: print(r.stderr[-2000:])
assert r.returncode == 0, "R package install failed"
print(f"install finished in {time.time()-t0:.1f} s")

## Minimal input and code acquisition with provenance / 最小入力とコードの取得（出典付き）

All bytes come from the authors' public repository at the **pinned commit** `63f73a33284c83faa0ef356daa88118d1dfa4b46`:

- `04_models/run_revised_table2_rf_svm_cv_holdout_models.R` — the Table 2 runner (used verbatim),
- `04_models/revised_table2_rf_svm_model_specs.csv` — the model specification table,
- `input_data/training_90.csv` (90 % grouped training partition), `input_data/holdout_test_10.csv` (untouched 10 % holdout), `input_data/training_90_grouped5fold_unit_assignments.csv` (unit-level fold assignment), `input_data/input_dataset_summary.csv` (author-provided row/unit summary).

Each file is checked against its expected byte size and **git blob SHA-1** recorded from the GitHub tree at the pinned commit, so a truncated or altered download fails loudly. Total ≈ 6.6 MB, stored under `/content/rice-blb/` in the layout the script expects (`<bundle>/04_models/…`, `<bundle>/input_data/…`).

**参考:** 著者リポジトリの固定コミットから、Table 2 実行スクリプトとその仕様 CSV、学習/ホールドアウト用の公開入力テーブルを取得し、サイズと git blob SHA-1 で完全性を検証します（合計約 6.6 MB）。

In [ ]:
import urllib.request, hashlib, pathlib
REPO = "ADSB96/rice-blb-multimodal-phenotyping"; COMMIT = "63f73a33284c83faa0ef356daa88118d1dfa4b46"
PINNED = {'04_models/run_revised_table2_rf_svm_cv_holdout_models.R': (52819, '7877e64997ddfd79b6e2b898103d7905f84fd5e5'), '04_models/revised_table2_rf_svm_model_specs.csv': (3018, 'c88851626683411eb8e869db1762fdb396a4b1ab'), 'input_data/training_90.csv': (5908836, 'c907ec31813ee2981e7a11831b0701275bb05fba'), 'input_data/holdout_test_10.csv': (657461, '581cae9646de3fd6ef3e15b8e03c1d084fc50f9b'), 'input_data/training_90_grouped5fold_unit_assignments.csv': (1921, '7353cdc5405c993b96c7a26a1ca2c02f487f567c'), 'input_data/input_dataset_summary.csv': (186, 'adf78a60b3cd46189f9f96e16a9c0bb771a48bb9')}
root = pathlib.Path("/content/rice-blb")
for rel, (size, blob) in PINNED.items():
    dest = root / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{rel}"
    if not dest.exists() or dest.stat().st_size != size:
        urllib.request.urlretrieve(url, dest)
    data = dest.read_bytes()
    assert len(data) == size, (rel, len(data), size)
    sha = hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()
    assert sha == blob, (rel, sha, blob)
    print(f"OK {rel}: {len(data)} bytes, git-blob {sha[:12]}")
print("all pinned files verified")

## Input preview / 入力データのプレビュー

Before any modelling, inspect the actual inputs: the author-provided dataset summary, the model spec table (which defines the exact `model_id` strings), dataset dimensions, the number of distinct microplot units (the leakage-safeguard groups), the 4-class and 6-class severity label distributions, and the grouped 5-fold assignment counts. All numbers shown here are read from the downloaded files inside Colab — nothing is assumed from the paper text.

**参考:** モデル実行前に、著者提供のデータサマリ・モデル仕様・行数/ユニット数・ラベル分布・fold 割当を実ファイルから確認します。

In [ ]:
r_code = r"""
library(data.table)
train <- fread("/content/rice-blb/input_data/training_90.csv")
test <- fread("/content/rice-blb/input_data/holdout_test_10.csv")
spec <- fread("/content/rice-blb/04_models/revised_table2_rf_svm_model_specs.csv")
assign <- fread("/content/rice-blb/input_data/training_90_grouped5fold_unit_assignments.csv")
summary_csv <- fread("/content/rice-blb/input_data/input_dataset_summary.csv")
cat("== author-provided dataset summary ==\n"); print(summary_csv)
cat("== model spec (MODEL_IDS come from model_id) ==\n"); print(spec)
cat("train dims:", nrow(train), "x", ncol(train), "  test dims:", nrow(test), "x", ncol(test), "\n")
cat("units train/test:", uniqueN(train$unit_norm), uniqueN(test$unit_norm), " overlap:", length(intersect(train$unit_norm, test$unit_norm)), "\n")
cat("BB_rating (6-class) train:\n"); print(table(train$BB_rating))
cat("BB_rating_4class train:\n"); print(table(train$BB_rating_4class))
cat("cv_fold train:\n"); print(table(train$cv_fold, useNA="ifany"))
bin_cols <- grep("_bin[1-6]$", names(train), value = TRUE)
cluster_cols <- grep("^cluster_[0-9]+_ratio$", names(train), value = TRUE)
struct_cols <- grep("^structural_", names(train), value = TRUE)
cat("feature groups -> binned VI:", length(bin_cols), " cluster-ratio:", length(cluster_cols), " structural:", length(struct_cols), "\n")
"""
r = subprocess.run(["Rscript", "-e", r_code], capture_output=True, text=True, timeout=600)
print(r.stdout)
if r.returncode: print(r.stderr[-1500:])
assert r.returncode == 0

## Model selection for this demonstration / 本デモで実行するモデル

The paper's abstract and Table 2 identify **Random Forest on binned vegetation indices (4 severity categories)** as the strongest configuration (reported holdout accuracy 0.762, weighted F1 0.760, QWK 0.859). We select the spec rows with `severity_categories == 4` and `feature_key == "binned_vi"` **from the spec file itself** (no hand-typed model ids), so the notebook stays valid if the spec's `model_id` strings change. To run more of the Table 2 suite, extend the filter (e.g. add `cluster_plus_binned`, or drop the severity filter) — expect proportionally longer runtime.

**参考:** 論文の最良構成「binned VI × RandomForest（4 クラス）」を仕様 CSV から動的に選択します（`MODEL_IDS` は手入力ではなくファイル由来）。フィルタを広げれば Table 2 の他モデルも実行できます（時間は比例して増加）。

In [ ]:
r_code = r"""
library(data.table)
spec <- fread("/content/rice-blb/04_models/revised_table2_rf_svm_model_specs.csv")
sel <- spec[severity_categories == 4 & feature_key == "binned_vi"]
cat("selected model_ids:", paste(sel$model_id, collapse = ", "), "\n")
writeLines(paste(sel$model_id, collapse = ","), "/content/rice-blb/selected_model_ids.txt")
"""
r = subprocess.run(["Rscript", "-e", r_code], capture_output=True, text=True, timeout=300)
print(r.stdout)
if r.returncode: print(r.stderr[-800:])
assert r.returncode == 0

## Run the authors' Table 2 pipeline / 著者の Table 2 パイプラインを実行

This cell launches the **unmodified** author script with `Rscript`, streaming its log. The script itself: reads the two published partitions, asserts **no unit overlap** between train and holdout (`stop("Unit overlap detected…")` otherwise), attaches the grouped 5-fold assignment, and per selected model runs nested grouped CV (SMOTE on training portions only; 9-point tuning grid × 3 inner folds per outer fold), then trains a consensus-parameter final model on the full 90 % partition and evaluates once on the untouched 10 % holdout. It writes per-fold metrics, ordinal metrics (quadratic weighted Cohen's kappa `QWK`, mean absolute error in class units `MAE`), confusion matrices, predictions and `table2_model_summary_detailed.csv` under `/content/rice-blb/outputs/…`.

**参考:** 著者スクリプトを無変更で `Rscript` により実行し、ログをストリーム表示します。グループ化 5 折 CV（SMOTE は学習部分のみ）＋未接触ホールドアウト評価が行われ、指標・混同行列・予測が `/content/rice-blb/outputs/` に保存されます。

In [ ]:
import subprocess, time, os, pathlib
bundle = "/content/rice-blb"
env = dict(os.environ)
env["MODEL_IDS"] = pathlib.Path(bundle + "/selected_model_ids.txt").read_text().strip()
env["MODEL_OUTPUT_ROOT"] = bundle + "/outputs/table2_demo"
t0 = time.time()
proc = subprocess.run(
    ["Rscript", f"{bundle}/04_models/run_revised_table2_rf_svm_cv_holdout_models.R"],
    cwd=bundle, env=env, text=True, timeout=5400,
)
print(f"Rscript exit code: {proc.returncode} in {(time.time()-t0)/60:.1f} min")
assert proc.returncode == 0, "author script failed - see log above"
out = pathlib.Path(env["MODEL_OUTPUT_ROOT"])
print("outputs written to:", out)
print(sorted(p.name for p in out.iterdir()))

## Results — executed metrics / 実行結果の指標

`table2_model_summary_detailed.csv` aggregates, per model: grouped-CV metric mean/SD/95 % CI (accuracy, weighted precision/recall/F1, macro-F1, QWK, MAE) and the single holdout evaluation. Display it below and export a copy next to the outputs.

**参考:** 著者スクリプトが出力したサマリ CSV（CV 平均±SD・95%CI とホールドアウト評価）を表示します。

In [ ]:
import pandas as pd, pathlib
out = pathlib.Path("/content/rice-blb/outputs/table2_demo")
summary = pd.read_csv(out / "table2_model_summary_detailed.csv")
show_cols = [c for c in [
    "model_name", "severity_categories", "algorithm", "feature_label",
    "holdout_accuracy", "holdout_f1_weighted", "holdout_qwk", "holdout_mae_class",
    "cv_accuracy_mean", "cv_accuracy_sd", "cv_qwk_mean", "cv_mae_class_mean"] if c in summary.columns]
display(summary[show_cols])
summary.to_csv("/content/rice_blb_table2_summary_export.csv", index=False)
print("exported: /content/rice_blb_table2_summary_export.csv")

### Holdout confusion matrix of the selected model / 選択モデルのホールドアウト混同行列

The final model's one-shot holdout predictions (`final_holdout_predictions.csv`) are summarized as a confusion matrix, shown in class units (the paper's `BB_rating_4class` levels). Off-diagonal mass between neighbouring severity classes is the paper's dominant error pattern; this figure is generated here from the executed run (not an author figure).

**参考:** 最終モデルのホールドアウト予測から混同行列を作図します（このノートブックで実行した結果に基づく追加可視化であり、論文の図ではありません）。

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt
import numpy as np
out = pathlib.Path("/content/rice-blb/outputs/table2_demo")
model_dirs = [p for p in out.iterdir() if p.is_dir()]
assert model_dirs, "no model output directory found"
preds = pd.read_csv(model_dirs[0] / "final_holdout_predictions.csv")
labels = sorted(preds["actual"].unique().tolist())
cm = pd.crosstab(preds["actual"], preds["predicted"]).reindex(index=labels, columns=labels, fill_value=0)
fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=120)
im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(len(labels)), [str(x) for x in labels])
ax.set_yticks(range(len(labels)), [str(x) for x in labels])
ax.set_xlabel("Predicted BB_rating_4class"); ax.set_ylabel("Actual BB_rating_4class")
ax.set_title("Holdout confusion matrix - binned-VI RandomForest (this run)")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm.values[i, j]), ha="center", va="center",
                color="white" if cm.values[i, j] > cm.values.max()/2 else "black")
fig.colorbar(im, ax=ax, label="holdout rows")
fig.tight_layout(); fig.savefig("/content/rice_blb_holdout_confusion.png"); plt.show()
print("rows:", len(preds), " classes:", labels)

### Fold-level CV accuracy of the executed model / 実行モデルの fold ごと CV 精度

Per-fold accuracy from `cv_fold_metrics.csv` makes the grouped-CV spread visible (the paper reports CV accuracy mean ± SD). Bars are per-outer-fold accuracy from this executed run; the dashed line marks their mean.

**参考:** グループ化 CV の fold ごと精度を棒グラフにします（本実行の実測値、破線は平均）。

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt
out = pathlib.Path("/content/rice-blb/outputs/table2_demo")
model_dirs = [p for p in out.iterdir() if p.is_dir()]
fm = pd.read_csv(model_dirs[0] / "cv_fold_metrics.csv")
fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=120)
ax.bar(fm["fold"].astype(str), fm["accuracy"], color="#4878b0")
ax.axhline(fm["accuracy"].mean(), ls="--", c="gray", label=f"mean {fm['accuracy'].mean():.3f}")
ax.set_xlabel("grouped CV outer fold"); ax.set_ylabel("accuracy")
ax.set_title("Grouped 5-fold CV accuracy - binned-VI RandomForest (this run)")
ax.legend(); fig.tight_layout(); fig.savefig("/content/rice_blb_cv_fold_accuracy.png"); plt.show()
print(fm[["fold", "n_train", "n_valid", "accuracy", "macro_f1"]].to_string(index=False))

## Comparison with the paper's Table 2 (reference values) / 論文 Table 2 との比較

Reference values below are transcribed from Table 2 of the verified open-access full text (OpenAlex-cached publisher PDF, work `W7213899941`): *binned VI, 4 categories, RandomForest — holdout accuracy 0.762, holdout F1 0.760, holdout QWK 0.859, holdout MAE 0.277, CV accuracy 0.667 ± 0.031*. The cell compares them with the executed run; **exact agreement is not expected** — the authors' published numbers came from a specific historical run and library versions, while grouped-CV fold composition and tie-breaking can shift results. Agreement within a few percentage points indicates the published pipeline reproduces.

**参考:** 比較対象の参考値は検証済み全文（OpenAlex キャッシュの出版社 PDF）Table 2 からの書き写しです。実行値はライブラリ版やタイブレークの影響で数ポイント程度変動し得ます。

In [ ]:
import pandas as pd, pathlib
out = pathlib.Path("/content/rice-blb/outputs/table2_demo")
model_dirs = [p for p in out.iterdir() if p.is_dir()]
summary = pd.read_csv(out / "table2_model_summary_detailed.csv")
row = summary.iloc[0]
ref = {"holdout_accuracy": 0.762, "holdout_f1_weighted": 0.760, "holdout_qwk": 0.859,
       "holdout_mae_class": 0.277, "cv_accuracy_mean": 0.667}
cmp = pd.DataFrame({
    "metric": list(ref.keys()),
    "paper Table 2": list(ref.values()),
    "this run": [float(row[c]) if c in row and pd.notna(row[c]) else float("nan") for c in ref],
})
cmp["delta"] = cmp["this run"] - cmp["paper Table 2"]
display(cmp.round(4))

## Validation record, limitations, and references / 検証記録・限界・参照

**Scope actually executed:** the authors' pinned Table 2 runner on the published de-identified input tables, for the 4-class binned-VI model subset — grouped 5-fold CV on the 90 % partition plus one holdout evaluation. Not executed here: the remaining Table 2 rows (other feature sets / SVM / XGBoost / 6-class), the supplementary cluster-ratio granularity comparison (`8C/12C/16C`), and any raw point-cloud or multispectral processing (raw sensor data are not published; only derived feature tables are).

**Known caveats:** the repository carries no LICENSE file (public for reproduction; reuse terms should be verified with the authors); package versions on the Colab VM are printed above and may differ from the authors' historical environment; `colab` runtimes may vary in CPU speed, so wall-time differs between sessions.

**References**
- S. Baruah et al., *A practical framework for rice bacterial blight severity classification using 3D multispectral sensing*, Smart Agricultural Technology 15 (2026) 102586, DOI 10.1016/j.atech.2026.102586 (verified full text via OpenAlex work W7213899941).
- Author code: github.com/ADSB96/rice-blb-multimodal-phenotyping @ `63f73a33284c83faa0ef356daa88118d1dfa4b46` (`04_models/run_revised_table2_rf_svm_cv_holdout_models.R`, README §"Key Modeling Assumptions").
- Input tables: same commit, `input_data/` (sizes and git-blob SHA-1s verified in-cell).

**参考:** 本ノートブックは Table 2 のうち binned-VI×RandomForest（4 クラス）部分を実行します。他の特徴量・アルゴリズム・6 クラス設定や補足のクラスタ粒度比較は未実行です。生の点群/マルチスペクトル処理は対象外（生データ未公開、派生特徴量テーブルのみ公開）。リポジトリに LICENSE がない点に留意してください。